<a href="https://colab.research.google.com/github/valceven/AliacSearchAlgo/blob/master/CNNAct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
print("wowza")

wowza


In [6]:
!git clone https://huggingface.co/datasets/seaurkin/facial_exrpressions

Cloning into 'facial_exrpressions'...
remote: Enumerating objects: 9364, done.
remote: Total 9364 (delta 0), reused 0 (delta 0), pack-reused 9364 (from 1)
Receiving objects: 100% (9364/9364), 1.50 MiB | 5.08 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Updating files: 100% (9365/9365), done.
Filtering content: 100% (9363/9363), 17.88 MiB | 85.00 KiB/s, done.


In [20]:
!ls ../content/

facial_exrpressions  sample_data


In [17]:
import tensorflow
from tensorflow import keras
from keras.layers import Dense, GlobalAveragePooling2D, Conv2D, MaxPooling2D
from keras.optimizers import Adam
from keras.models import Model
from keras.applications import ResNet50V2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input
import numpy as np
import pandas as pd
import requests
import os
from tqdm import tqdm
import torch
from PIL import Image
from sklearn.model_selection import train_test_split

In [23]:
dataset_path = "/content/facial_exrpressions/FACS"
labels = os.listdir(dataset_path)
print(labels)

['raisedbrows', 'frowning', 'smile', 'openmouth', 'neutral', 'kiss']


In [53]:
data = {"filename": [], "label": []};

for label in labels:
  label_path = os.path.join(dataset_path, label);
  if os.path.isdir(label_path):
    for file in os.listdir(label_path):
      if file.endswith((".jpg", ".png")):
        data["filename"].append(os.path.join(label_path, file))
        data["label"].append(label);

df = pd.DataFrame(data)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.head()

from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["label"])
print(df["label"].unique())

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train: {len(train_df)}")
print(f"Val: {len(val_df)}")
print(f"Test: {len(test_df)}")

[0 5 1 2 4 3]
Train: 6554
Val: 1404
Test: 1405


In [54]:
train_df["label"] = train_df["label"].astype(str)
val_df["label"] = val_df["label"].astype(str)
test_df["label"] = test_df["label"].astype(str)

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
)

train_batches = datagen.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="label",
    target_size=(224, 224),
    batch_size=32,
    class_mode="sparse",
    shuffle=True
)

valid_batches = datagen.flow_from_dataframe(
    val_df,
    x_col="filename",
    y_col="label",
    target_size=(224, 224),
    batch_size=32,
    class_mode="sparse",
    shuffle=True
)

test_batches = datagen.flow_from_dataframe(
    test_df,
    x_col="filename",
    y_col="label",
    target_size=(224, 224),
    batch_size=32,
    class_mode="sparse",
)

print(f"Train: {len(train_batches)}")
print(f"Val: {len(valid_batches)}")
print(f"Test: {len(test_batches)}")

Found 6554 validated image filenames belonging to 6 classes.
Found 1404 validated image filenames belonging to 6 classes.
Found 1405 validated image filenames belonging to 6 classes.
Train: 205
Val: 44
Test: 44


In [55]:
base_model = ResNet50V2(weights="imagenet",include_top=False, input_shape=(224, 224, 3))
base_model.trainable = True

model = base_model.output
model = GlobalAveragePooling2D()(model)
model = Dense(128, activation="relu")(model)
output_layer = Dense(len(labels), activation="softmax")(model)

model = Model(inputs=base_model.input, outputs=output_layer)
model.summary()

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8             │ (None, 224, 224, 3)    │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_pad (ZeroPadding2D) │ (None, 230, 230, 3)    │              0 │ input_layer_8[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_conv (Conv2D)       │ (None, 112, 112, 64)   │          9,472 │ conv1_pad[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ pool1_pad (ZeroPadding2D) │ (None, 114, 114, 64)   │              0 │ conv1_conv[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ pool1_pool (MaxPooling2D) │ (None, 56, 56, 64)     │              0 │ pool1_pad[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_preact_bn    │ (None, 56, 56, 64)     │            256 │ pool1_pool[0][0]       │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_preact_relu  │ (None, 56, 56, 64)     │              0 │ conv2_block1_preact_b… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_conv       │ (None, 56, 56, 64)     │          4,096 │ conv2_block1_preact_r… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_bn         │ (None, 56, 56, 64)     │            256 │ conv2_block1_1_conv[0… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_relu       │ (None, 56, 56, 64)     │              0 │ conv2_block1_1_bn[0][… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_pad        │ (None, 58, 58, 64)     │              0 │ conv2_block1_1_relu[0… │
│ (ZeroPadding2D)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_conv       │ (None, 56, 56, 64)     │         36,864 │ conv2_block1_2_pad[0]… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_bn         │ (None, 56, 56, 64)     │            256 │ conv2_block1_2_conv[0… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_relu       │ (None, 56, 56, 64)     │              0 │ conv2_block1_2_bn[0][… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_0_conv  

 Total params: 23,827,846 (90.90 MB)

 Trainable params: 23,782,406 (90.72 MB)

 Non-trainable params: 45,440 (177.50 KB)

In [58]:
model.compile(optimizer=Adam(learning_rate=0.001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_batches, epochs=10, validation_data=valid_batches, batch_size=64)

Epoch 1/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 175s 639ms/step - accuracy: 0.9077 - loss: 0.2810 - val_accuracy: 0.9038 - val_loss: 0.2785
Epoch 2/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 111s 541ms/step - accuracy: 0.9105 - loss: 0.2620 - val_accuracy: 0.8148 - val_loss: 0.5357
Epoch 3/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 105s 513ms/step - accuracy: 0.9150 - loss: 0.2386 - val_accuracy: 0.8640 - val_loss: 0.4156
Epoch 4/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 107s 521ms/step - accuracy: 0.9189 - loss: 0.2277 - val_accuracy: 0.8789 - val_loss: 0.3696
Epoch 5/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 107s 523ms/step - accuracy: 0.9167 - loss: 0.2347 - val_accuracy: 0.8981 - val_loss: 0.2917
Epoch 6/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 107s 520ms/step - accuracy: 0.9254 - loss: 0.2089 - val_accuracy: 0.8013 - val_loss: 0.6571
Epoch 7/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 107s 522ms/step - accuracy: 0.9182 - loss: 0.2216 - val_accuracy: 0.7208 - val_loss: 0.8547
Epoch 8/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 105s 514ms/step - accuracy: 0.9339 -

In [59]:
model.evaluate(test_batches)

44/44 ━━━━━━━━━━━━━━━━━━━━ 19s 427ms/step - accuracy: 0.8982 - loss: 0.3006


[0.28173819184303284, 0.9067615866661072]

In [62]:
model.save("facial_expression_model.h5")

In [65]:
!ls

from google.colab import files
files.download("facial_expression_model.h5")

facial_expression_model.h5  facial_exrpressions  sample_data


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>